# ClusterJudge – Comparing Clusterings via Noisy Pairwise Judgements

Fully reproducible notebook (Google Colab compatible).  
All figures saved to `figures/`, logs to `logs/`.

## Cell 1 – Setup

In [ ]:
# ── Install extra packages (uncomment on Colab) ──────────────────────────────
# !pip install -q networkx scipy seaborn

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import ConnectionPatch
import seaborn as sns
import networkx as nx
from scipy.linalg import orthogonal_procrustes
import warnings
import random
import os
import logging

warnings.filterwarnings('ignore')

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED = 42        # Global random seed; change to get a different (but reproducible) realisation

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Hardware / dtype ──────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype  = torch.float32

# ── Output directories ────────────────────────────────────────────────────────
os.makedirs('figures', exist_ok=True)
os.makedirs('logs',    exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename='logs/run.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    force=True,
)
logging.info('Session started. Device: %s', device)

# ── Seaborn global style ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
PALETTE     = sns.color_palette('tab10')          # discrete colours for clusters / judgements
GRAY_SHADES = ['#333333', '#777777', '#BBBBBB']   # background grey tones for judgement plot

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
print('Setup complete.')


## Cell 2 – Dataset Generation

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
N_PER_CLUSTER = 10    # Points per cluster; more -> denser, smoother clusters
N_CLUSTERS    = 3     # Number of ground-truth clusters; must match len(CLUSTER_PARAMS)

CLUSTER_PARAMS = [
    {'mean': np.array([ 0.0,  0.0]), 'std': 1.0, 'label': 0},  # broad, centred; increase std -> more overlap
    {'mean': np.array([ 2.0,  2.0]), 'std': 0.3, 'label': 1},  # tight, top-right; decrease std -> tighter
    {'mean': np.array([-5.0, -5.0]), 'std': 0.5, 'label': 2},  # bottom-left; move mean closer -> harder to separate
]

MARKERS        = ['^', 'o', 'v']                              # triangle-up, circle, triangle-down
CLUSTER_COLORS = [PALETTE[0], PALETTE[1], PALETTE[2]]        # seaborn tab10 colours

# ── Generate data ─────────────────────────────────────────────────────────────
np.random.seed(SEED)

data_points = []
labels      = []
for cp in CLUSTER_PARAMS:
    pts = np.random.randn(N_PER_CLUSTER, 2) * cp['std'] + cp['mean']
    data_points.append(pts)
    labels.extend([cp['label']] * N_PER_CLUSTER)

data_points = np.vstack(data_points)   # (N, 2)
labels      = np.array(labels)         # (N,)
N_POINTS    = len(data_points)

logging.info('Dataset: %d points, %d clusters', N_POINTS, N_CLUSTERS)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

for c, (cp, marker, col) in enumerate(zip(CLUSTER_PARAMS, MARKERS, CLUSTER_COLORS)):
    mask = labels == c
    ax.scatter(
        data_points[mask, 0], data_points[mask, 1],
        facecolors='none',
        edgecolors=col,
        marker=marker,
        s=90, linewidths=1.8,
        label=f'Cluster {c}  mu={cp["mean"]}',
    )

ax.set_title('Ground-Truth Dataset')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(frameon=False, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('figures/dataset.jpg', dpi=150)
plt.savefig('figures/dataset.pdf')
plt.show()

print(f'Data shape : {data_points.shape}')
print(f'Labels     : {labels}')


## Cell 3 – Judgement Generation

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
N_JUDGEMENTS = 20    # Number of triple comparisons; more -> richer graph, more signal (increase to >=50 for 30 points to improve coverage of C(30,2)=435 possible pairs)
TEMPERATURE  = 1.0   # Bradley-Terry temperature; higher -> more random choices, lower -> more deterministic

# ── Generate judgements ───────────────────────────────────────────────────────
np.random.seed(SEED)
random.seed(SEED)

judgements = []

for _ in range(N_JUDGEMENTS):
    # Pick 3 distinct random points
    idx          = np.random.choice(N_POINTS, size=3, replace=False)
    a_idx, b_idx, c_idx = int(idx[0]), int(idx[1]), int(idx[2])
    a, b, c      = data_points[a_idx], data_points[b_idx], data_points[c_idx]

    # Pairwise distances
    d_ab = float(np.linalg.norm(a - b))
    d_ac = float(np.linalg.norm(a - c))
    d_bc = float(np.linalg.norm(b - c))

    # Bradley-Terry: closer pair -> higher probability of being selected as 'same cluster'
    scores = np.array([
        np.exp(-d_ab / TEMPERATURE),
        np.exp(-d_ac / TEMPERATURE),
        np.exp(-d_bc / TEMPERATURE),
    ])
    probs = scores / scores.sum()

    # The three candidate pairs
    pairs = [(a_idx, b_idx), (a_idx, c_idx), (b_idx, c_idx)]

    # Sample winner pair
    choice      = int(np.random.choice(3, p=probs))
    winner_pair = pairs[choice]
    loser_pairs = [p for i, p in enumerate(pairs) if i != choice]

    judgements.append({
        'a_idx':    a_idx,    'b_idx':   b_idx,    'c_idx':   c_idx,
        'winner_i': winner_pair[0], 'winner_j': winner_pair[1],
        'loser1_i': loser_pairs[0][0], 'loser1_j': loser_pairs[0][1],
        'loser2_i': loser_pairs[1][0], 'loser2_j': loser_pairs[1][1],
        'd_ab': d_ab, 'd_ac': d_ac, 'd_bc': d_bc,
        'prob_ab': probs[0], 'prob_ac': probs[1], 'prob_bc': probs[2],
    })

df_judgements = pd.DataFrame(judgements)
df_judgements.to_csv('judgements.csv', index=False)
logging.info('Saved %d judgements to judgements.csv', N_JUDGEMENTS)
print(f'Saved {len(df_judgements)} judgements -> judgements.csv')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

# Background scatter – three shades of grey, same markers as before
for c, (gray, marker) in enumerate(zip(GRAY_SHADES, MARKERS)):
    mask = labels == c
    ax.scatter(
        data_points[mask, 0], data_points[mask, 1],
        facecolors='none', edgecolors=gray,
        marker=marker, s=80, linewidths=1.5, zorder=3,
    )

# One distinct colour per judgement
jmt_colors = sns.color_palette('husl', N_JUDGEMENTS)

for jidx, row in df_judgements.iterrows():
    col = jmt_colors[jidx]
    wi, wj   = int(row['winner_i']), int(row['winner_j'])
    l1i, l1j = int(row['loser1_i']), int(row['loser1_j'])
    l2i, l2j = int(row['loser2_i']), int(row['loser2_j'])

    # Solid edge – winner pair (likely same cluster)
    ax.plot(
        [data_points[wi, 0], data_points[wj, 0]],
        [data_points[wi, 1], data_points[wj, 1]],
        color=col, linewidth=1.8, alpha=0.9, zorder=2,
    )
    # Dashed edges – loser pairs (likely different cluster)
    for li, lj in [(l1i, l1j), (l2i, l2j)]:
        ax.plot(
            [data_points[li, 0], data_points[lj, 0]],
            [data_points[li, 1], data_points[lj, 1]],
            color=col, linewidth=0.8, alpha=0.28, linestyle='--', zorder=1,
        )

ax.set_title('Judgements\n(solid = likely same cluster, dashed = likely different)')
ax.set_xlabel('x')
ax.set_ylabel('y')
sns.despine()
plt.tight_layout()
plt.savefig('figures/judgements.jpg', dpi=150)
plt.savefig('figures/judgements.pdf')
plt.show()


## Cell 4 – Cluster Generation (Max Cut on Solid Edges)

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
K_CLUSTERS   = 3    # Number of clusters; more -> finer partition, harder to optimise
N_RESTARTS   = 30   # Greedy restarts; more -> better chance of finding a good max-cut
MAX_ITER_CUT = 200  # Max greedy iterations per restart; more -> tighter convergence

# ── Read judgements ───────────────────────────────────────────────────────────
df_j = pd.read_csv('judgements.csv')

# ── Build solid-edge graph (winner pairs) ─────────────────────────────────────
G_solid = nx.Graph()
G_solid.add_nodes_from(range(N_POINTS))

for _, row in df_j.iterrows():
    wi, wj = int(row['winner_i']), int(row['winner_j'])
    if G_solid.has_edge(wi, wj):
        G_solid[wi][wj]['weight'] += 1
    else:
        G_solid.add_edge(wi, wj, weight=1)

print(f'Solid graph: {G_solid.number_of_nodes()} nodes, {G_solid.number_of_edges()} edges')

# ── Greedy Max-Cut into K clusters ───────────────────────────────────────────
def _node_cut(G, assignment, n, cluster):
    # Returns weighted edges from node n to nodes NOT in cluster
    return sum(
        d.get('weight', 1)
        for nbr, d in G[n].items()
        if assignment[nbr] != cluster
    )

def greedy_maxcut(G, k, n_restarts=N_RESTARTS, max_iter=MAX_ITER_CUT, seed=SEED):
    rng          = np.random.default_rng(seed)
    nodes        = list(G.nodes())
    best_assign  = None
    best_cut_val = -1

    for _ in range(n_restarts):
        assignment = {n: int(rng.integers(0, k)) for n in nodes}

        for _ in range(max_iter):
            improved   = False
            node_order = list(nodes)
            rng.shuffle(node_order)

            for n in node_order:
                cur_cluster    = assignment[n]
                cur_cut        = _node_cut(G, assignment, n, cur_cluster)
                best_c, best_c_cut = cur_cluster, cur_cut

                for c in range(k):
                    if c == cur_cluster:
                        continue
                    cut = _node_cut(G, assignment, n, c)
                    if cut > best_c_cut:
                        best_c_cut, best_c = cut, c

                if best_c != cur_cluster:
                    assignment[n] = best_c
                    improved = True

            if not improved:
                break

        total_cut = sum(
            d.get('weight', 1)
            for u, v, d in G.edges(data=True)
            if assignment[u] != assignment[v]
        )
        if total_cut > best_cut_val:
            best_cut_val = total_cut
            best_assign  = dict(assignment)

    return best_assign, best_cut_val

maxcut_assignment, maxcut_val = greedy_maxcut(G_solid, K_CLUSTERS)
print(f'Max-Cut value : {maxcut_val}')
logging.info('Max-Cut: k=%d, cut_value=%d', K_CLUSTERS, maxcut_val)

maxcut_labels = np.array([maxcut_assignment.get(i, 0) for i in range(N_POINTS)])

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

cluster_colors_mc = [PALETTE[3], PALETTE[4], PALETTE[5]]

for c in range(K_CLUSTERS):
    mask = maxcut_labels == c
    ax.scatter(
        data_points[mask, 0], data_points[mask, 1],
        facecolors='none', edgecolors=cluster_colors_mc[c],
        marker=MARKERS[c % len(MARKERS)], s=90, linewidths=1.8,
        label=f'Cluster {c}',
    )

ax.set_title(f'Max-Cut Clustering (k={K_CLUSTERS}, cut={maxcut_val})')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(frameon=False, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('figures/maxcut_clustering.jpg', dpi=150)
plt.savefig('figures/maxcut_clustering.pdf')
plt.show()


## Cell 5 – Cluster Generation II (Min Cut on Dashed Edges)

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
K_MIN = 3    # Target number of partitions; hierarchical Stoer-Wagner splits K_MIN-1 times

# ── Build dashed-edge graph (loser pairs) ─────────────────────────────────────
G_dashed = nx.Graph()
G_dashed.add_nodes_from(range(N_POINTS))

for _, row in df_j.iterrows():
    for (li, lj) in [(int(row['loser1_i']), int(row['loser1_j'])),
                     (int(row['loser2_i']), int(row['loser2_j']))]:
        if G_dashed.has_edge(li, lj):
            G_dashed[li][lj]['weight'] += 1
        else:
            G_dashed.add_edge(li, lj, weight=1)

print(f'Dashed graph: {G_dashed.number_of_nodes()} nodes, {G_dashed.number_of_edges()} edges')

# ── Hierarchical Min-Cut (Stoer-Wagner) into K partitions ─────────────────────
def hierarchical_mincut(G, k):
    # Repeatedly split the largest connected sub-graph using global min-cut
    # until we have k partitions. Uses networkx.stoer_wagner.
    groups = [list(G.nodes())]

    for _ in range(k - 1):
        candidates = [
            (grp, G.subgraph(grp))
            for grp in groups
            if G.subgraph(grp).number_of_edges() > 0
        ]
        if not candidates:
            break

        target_grp, target_sg = max(candidates, key=lambda x: x[1].number_of_edges())

        try:
            _, partition = nx.stoer_wagner(target_sg)
            S = list(partition[0])
            T = list(partition[1])
        except Exception:
            nodes = list(target_sg.nodes())
            mid   = max(1, len(nodes) // 2)
            S, T  = nodes[:mid], nodes[mid:]

        groups.remove(target_grp)
        if S:
            groups.append(S)
        if T:
            groups.append(T)

    assignment = {}
    for cid, grp in enumerate(groups):
        for n in grp:
            assignment[n] = cid
    for n in G.nodes():
        if n not in assignment:
            assignment[n] = len(groups)
            groups.append([n])

    return assignment

mincut_assignment = hierarchical_mincut(G_dashed, K_MIN)
mincut_labels     = np.array([mincut_assignment.get(i, 0) for i in range(N_POINTS)])

mincut_val = sum(
    d.get('weight', 1)
    for u, v, d in G_dashed.edges(data=True)
    if mincut_assignment.get(u) != mincut_assignment.get(v)
)
print(f'Min-Cut value : {mincut_val}')
logging.info('Min-Cut (dashed): k=%d, cut_value=%d', K_MIN, mincut_val)

pd.DataFrame({
    'point_idx':    range(N_POINTS),
    'mincut_label': mincut_labels,
    'x':            data_points[:, 0],
    'y':            data_points[:, 1],
}).to_csv('mincut_clusters.csv', index=False)
print('Saved mincut_clusters.csv')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

cluster_colors_mc2 = [PALETTE[6], PALETTE[7], PALETTE[8]]

for c in range(K_MIN):
    mask = mincut_labels == c
    ax.scatter(
        data_points[mask, 0], data_points[mask, 1],
        facecolors='none', edgecolors=cluster_colors_mc2[c],
        marker=MARKERS[c % len(MARKERS)], s=90, linewidths=1.8,
        label=f'Cluster {c}',
    )

ax.set_title(f'Min-Cut Clustering (k={K_MIN}, cut={mincut_val})')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(frameon=False, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('figures/mincut_clustering.jpg', dpi=150)
plt.savefig('figures/mincut_clustering.pdf')
plt.show()


## Cell 6 – Meta Graph

In [ ]:
# ── Helper: ground-truth same-cluster check ──────────────────────────────────
def same_cluster(i, j):
    return int(labels[i]) == int(labels[j])

# Seaborn green / red / blue from 'deep' palette
_deep          = sns.color_palette('deep')
COLOR_GREEN    = _deep[2]   # green  – pair from same GT cluster
COLOR_RED      = _deep[3]   # red    – pair from different GT clusters
COLOR_CONFLICT = _deep[0]   # blue   – conflicting edge (same->diff)

# ── Build directed meta graph ─────────────────────────────────────────────────
MG = nx.DiGraph()

for _, row in df_judgements.iterrows():
    wi, wj   = int(row['winner_i']), int(row['winner_j'])
    l1i, l1j = int(row['loser1_i']), int(row['loser1_j'])
    l2i, l2j = int(row['loser2_i']), int(row['loser2_j'])

    # Canonical pair (smaller index first)
    winner = (min(wi, wj),   max(wi, wj))
    loser1 = (min(l1i, l1j), max(l1i, l1j))
    loser2 = (min(l2i, l2j), max(l2i, l2j))

    for node in [winner, loser1, loser2]:
        if node not in MG:
            MG.add_node(node, same=same_cluster(*node))

    # Directed edges: loser -> winner (arrow points to the closer / better pair)
    MG.add_edge(loser1, winner)
    MG.add_edge(loser2, winner)

print(f'Meta graph: {MG.number_of_nodes()} nodes, {MG.number_of_edges()} edges')
logging.info('Meta graph: %d nodes, %d edges', MG.number_of_nodes(), MG.number_of_edges())

# ── Node / edge colours ───────────────────────────────────────────────────────
node_same   = {n: MG.nodes[n]['same'] for n in MG.nodes()}
node_colors = [COLOR_GREEN if node_same[n] else COLOR_RED for n in MG.nodes()]

edge_colors = []
for u, v in MG.edges():
    if node_same[u] and not node_same[v]:
        edge_colors.append(COLOR_CONFLICT)  # conflicting: same -> diff
    else:
        edge_colors.append('#888888')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

pos = nx.spring_layout(MG, seed=SEED, k=2.5)

nx.draw_networkx_nodes(
    MG, pos, ax=ax,
    node_color=node_colors, node_size=500, alpha=0.9,
)
nx.draw_networkx_edges(
    MG, pos, ax=ax,
    edge_color=edge_colors, arrows=True,
    arrowsize=15, width=1.2, alpha=0.7,
    connectionstyle='arc3,rad=0.1',
)
nx.draw_networkx_labels(
    MG, pos, ax=ax,
    labels={n: f'({n[0]},{n[1]})' for n in MG.nodes()},
    font_size=6,
)

legend_handles = [
    mpatches.Patch(color=COLOR_GREEN,    label='Same GT cluster'),
    mpatches.Patch(color=COLOR_RED,      label='Different GT cluster'),
    mpatches.Patch(color=COLOR_CONFLICT, label='Conflicting edge (same->diff)'),
]
ax.legend(handles=legend_handles, frameon=False, fontsize=9, loc='upper left')
ax.set_title('Meta Graph of Pairwise Judgements')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/meta_graph.jpg', dpi=150)
plt.savefig('figures/meta_graph.pdf')
plt.show()


## Cell 7 – Learn Embedding via Gradient Descent

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
DIM_EMBED  = 2       # Embedding dimension; increase for richer representation, harder to visualise
LR         = 0.05    # Adam learning rate; higher -> faster but noisier convergence
N_EPOCHS   = 2000    # Training epochs; more -> closer to maximum-likelihood solution
BATCH_SIZE = 10      # Judgements per batch; smaller -> noisier gradient, more updates per epoch
LOG_EVERY  = 200     # Print loss every LOG_EVERY epochs

# ── Prepare index tensors ─────────────────────────────────────────────────────
torch.manual_seed(SEED)

winner_i_t = torch.tensor(df_judgements['winner_i'].values, dtype=torch.long, device=device)
winner_j_t = torch.tensor(df_judgements['winner_j'].values, dtype=torch.long, device=device)
loser1_i_t = torch.tensor(df_judgements['loser1_i'].values, dtype=torch.long, device=device)
loser1_j_t = torch.tensor(df_judgements['loser1_j'].values, dtype=torch.long, device=device)
loser2_i_t = torch.tensor(df_judgements['loser2_i'].values, dtype=torch.long, device=device)
loser2_j_t = torch.tensor(df_judgements['loser2_j'].values, dtype=torch.long, device=device)

n_judg = len(df_judgements)

# ── Learnable embeddings ──────────────────────────────────────────────────────
embeddings = nn.Parameter(
    torch.randn(N_POINTS, DIM_EMBED, device=device, dtype=dtype)
)
optimizer  = optim.Adam([embeddings], lr=LR)

# ── Batch log-likelihood (Bradley-Terry on distances) ─────────────────────────
def batch_log_likelihood(emb, wi, wj, l1i, l1j, l2i, l2j):
    d_win  = torch.norm(emb[wi]  - emb[wj],  dim=-1)   # (B,)
    d_los1 = torch.norm(emb[l1i] - emb[l1j], dim=-1)
    d_los2 = torch.norm(emb[l2i] - emb[l2j], dim=-1)
    log_Z  = torch.logsumexp(
        torch.stack([-d_win, -d_los1, -d_los2], dim=-1), dim=-1
    )
    return (-d_win - log_Z).mean()

# ── Training loop ─────────────────────────────────────────────────────────────
losses = []

for epoch in range(1, N_EPOCHS + 1):
    perm       = torch.randperm(n_judg, device=device)
    epoch_loss = 0.0
    n_batches  = 0

    for start in range(0, n_judg, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]

        optimizer.zero_grad()
        log_ll = batch_log_likelihood(
            embeddings,
            winner_i_t[idx], winner_j_t[idx],
            loser1_i_t[idx], loser1_j_t[idx],
            loser2_i_t[idx], loser2_j_t[idx],
        )
        loss = -log_ll
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    if epoch % LOG_EVERY == 0:
        print(f'Epoch {epoch:>5d}/{N_EPOCHS} | loss = {avg_loss:.4f}')
        logging.info('Epoch %d/%d  loss=%.4f', epoch, N_EPOCHS, avg_loss)

# ── Procrustes alignment (rotation only) ─────────────────────────────────────
learned_raw = embeddings.detach().cpu().numpy()
gt_centred  = data_points  - data_points.mean(axis=0)
le_centred  = learned_raw  - learned_raw.mean(axis=0)
R, _        = orthogonal_procrustes(le_centred, gt_centred)
learned_emb = le_centred @ R

# ── Training-loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses, color=PALETTE[0], linewidth=1.2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Neg. log-likelihood')
ax.set_title('Training Loss')
sns.despine()
plt.tight_layout()
plt.savefig('figures/training_loss.jpg', dpi=150)
plt.savefig('figures/training_loss.pdf')
plt.show()

# ── Comparison: learned (left) vs ground truth (right) ───────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 4.5))

for c, (marker, col) in enumerate(zip(MARKERS, CLUSTER_COLORS)):
    mask = labels == c
    ax_l.scatter(
        learned_emb[mask, 0], learned_emb[mask, 1],
        facecolors='none', edgecolors=col,
        marker=marker, s=90, linewidths=1.8,
        label=f'Cluster {c}',
    )
    ax_r.scatter(
        data_points[mask, 0], data_points[mask, 1],
        facecolors='none', edgecolors=col,
        marker=marker, s=90, linewidths=1.8,
        label=f'Cluster {c}',
    )

ax_l.set_title('Learned Embedding (Procrustes-aligned)')
ax_r.set_title('Ground Truth')
for ax in (ax_l, ax_r):
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax)

# Thin, low-opacity lines connecting matching points
for i in range(N_POINTS):
    con = ConnectionPatch(
        xyA=learned_emb[i], coordsA=ax_l.transData,
        xyB=data_points[i], coordsB=ax_r.transData,
        color='gray', alpha=0.15, linewidth=0.5, zorder=0,
    )
    fig.add_artist(con)

plt.tight_layout()
plt.savefig('figures/embedding_comparison.jpg', dpi=150, bbox_inches='tight')
plt.savefig('figures/embedding_comparison.pdf', bbox_inches='tight')
plt.show()

logging.info('Embedding learning complete.')
print('Embedding learning complete.')
